# Single-conv VadCLIP experiment
Keep LGT. Replace only the classifier with Conv1D(512, 128, k=5), GELU and channel-max kernel response. Train with Top-K video pooling from scratch; evaluate epoch 10. Run cells top to bottom.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
RUN_TAG = 'cnn_single_k5_c128_topk_s234'
RESULT_ROOT = PROJECT_ROOT / 'code' / 'results'
EPOCH_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_single_conv' / RUN_TAG
MODEL_PATH = EPOCH_DIR / 'epoch_010.pth'
TRAIN_DIR = RESULT_ROOT / 'train' / 'cnn_single_conv' / RUN_TAG
EVAL_DIR = RESULT_ROOT / 'evaluate' / 'cnn_single_conv' / RUN_TAG
ANALYZE_DIR = RESULT_ROOT / 'analyze' / 'cnn_single_conv' / RUN_TAG

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime first.')
if not SRC_DIR.is_dir():
    raise FileNotFoundError(SRC_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Missing /content/UCFClipFeatures and feature archive on Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
for required in (FEATURE_ROOT, SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
                 SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'test_cnn_single_conv.py',
                 LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
                 LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy'):
    if not required.exists():
        raise FileNotFoundError(required)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Run:', RUN_TAG)


## Smoke test
Check one Conv1D, padding, model outputs, gradients, and frozen CLIP before training.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'test_cnn_single_conv'], cwd=SRC_DIR, check=True)
sys.path.insert(0, str(SRC_DIR))
from model_cnn_vadclip import CLIPVADCNN
from ucf_cnn_common import TRAIN_LABEL_MAP, aggregate_video_logits, alignment_mil_loss, text_regularization_loss
from utils.tools import get_prompt_text

device = torch.device('cuda')
model = CLIPVADCNN(14, 512, 256, 512, 1, 2, 8, 10, 10, device,
                   kernel_size=5, hidden_width=128, score_mode='kernel_response',
                   cnn_architecture='single_conv').to(device)
visual = torch.randn(2, 256, 512, device=device)
lengths = torch.tensor([170, 256], device=device)
text, logits1, logits2 = model(visual, None, get_prompt_text(TRAIN_LABEL_MAP), lengths)
assert logits1.shape == (2, 256, 1) and logits2.shape == (2, 256, 14)
assert torch.equal(logits1[0, 170:], torch.zeros_like(logits1[0, 170:]))
video_logits = aggregate_video_logits(logits1, lengths, 'topk')
labels = torch.zeros(2, 14, device=device)
labels[0, 0], labels[1, 6] = 1, 1
loss = torch.nn.functional.binary_cross_entropy_with_logits(video_logits, torch.tensor([0., 1.], device=device))
loss = loss + alignment_mil_loss(logits2, labels, lengths) + text_regularization_loss(text)
assert torch.isfinite(loss)
loss.backward()
assert model.cnn_classifier.conv.weight.grad is not None
assert any(p.grad is not None for p in model.temporal.parameters())
assert all(p.grad is None for p in model.clipmodel.parameters())
print('Smoke test PASS:', tuple(logits1.shape), tuple(logits2.shape), float(loss))
del model
torch.cuda.empty_cache()


## Train and evaluate
Top-K pools valid snippet logits for video BCE; AUC/AP use sigmoid of every snippet logit. Outputs are separate from the earlier Max experiment.

In [ ]:
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Child process failed; traceback is printed above.')

config_path = TRAIN_DIR / 'config.json'
if MODEL_PATH.is_file():
    if not config_path.is_file():
        raise RuntimeError('Existing checkpoint has no training config; choose a new RUN_TAG.')
    saved = json.loads(config_path.read_text())
    expected = {'cnn_architecture': 'single_conv', 'kernel_size': 5, 'cnn_channels': 128,
                'score_mode': 'kernel_response', 'aggregation': 'topk', 'seed': 234,
                'batch_size': 64, 'lr': 1e-5, 'max_epoch': 10,
                'feature_root': str(FEATURE_ROOT), 'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
    if any(saved.get(key) != value for key, value in expected.items()):
        raise RuntimeError('Existing checkpoint config differs; choose a new RUN_TAG.')
    print('Matching checkpoint found; skipping training:', MODEL_PATH)
else:
    run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                 '--feature-root', str(FEATURE_ROOT), '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
                 '--cnn-architecture', 'single_conv', '--kernel-size', '5', '--cnn-channels', '128',
                 '--score-mode', 'kernel_response', '--aggregation', 'topk',
                 '--lambda-alignment', '1', '--lambda-text', '1', '--seed', '234',
                 '--batch-size', '64', '--lr', '1e-5', '--max-epoch', '10',
                 '--scheduler-milestones', '4', '8', '--num-workers', '2',
                 '--epoch-checkpoint-dir', str(EPOCH_DIR),
                 '--output-model-path', str(SRC_DIR / 'model' / 'cnn_single_conv' / f'{RUN_TAG}.pth'),
                 '--results-dir', str(TRAIN_DIR)])
metrics_path = EVAL_DIR / 'metrics.json'
if metrics_path.is_file():
    cached = json.loads(metrics_path.read_text())
    if cached.get('model_path') != str(MODEL_PATH) or cached.get('cnn_architecture') != 'single_conv':
        raise RuntimeError('Cached metrics use another model; choose a new RUN_TAG.')
else:
    run_command([sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                 '--model-type', 'cnn', '--model-path', str(MODEL_PATH),
                 '--cnn-architecture', 'single_conv', '--kernel-size', '5', '--cnn-channels', '128',
                 '--score-mode', 'kernel_response', '--train-config-json', str(config_path),
                 '--feature-root', str(FEATURE_ROOT),
                 '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                 '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                 '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                 '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                 '--evaluate-dir', str(EVAL_DIR), '--analyze-dir', str(ANALYZE_DIR)])
metrics = json.loads(metrics_path.read_text())
for key in ('classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP'):
    print(f'{key}: {metrics[key]:.2f}')
print('Train:', TRAIN_DIR, 'Evaluate:', EVAL_DIR, 'Analyze:', ANALYZE_DIR, sep='\n')
